## pass tool outputs to chat models
![alt text](https://python.langchain.com/assets/images/tool_invocation-7f277888701ee431a17607f1a035c080.png)
![alt text](https://python.langchain.com/assets/images/tool_results-71b4b90f33a56563c102d91e7821a993.png)

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(api_key="sk-762684b96deb4f748cb4383757f69a09",model="deepseek-chat" ,base_url="https://api.deepseek.com") 

In [3]:
from langchain_core.tools import tool


@tool
def add(a: int, b: int) -> int: #define a tool
    """Adds a and b."""
    return a + b


@tool
def multiply(a: int, b: int) -> int: #define another tool
    """Multiplies a and b."""
    return a * b


tools = [add, multiply] #list of tools

llm_with_tools = llm.bind_tools(tools) #bind tools to the llm

Humanmessage:message from a human
message = [SystemMessage(content="..."),HumanMessage(content="...")]
model=...
model.invoke(message)

In [61]:
from langchain_core.messages import HumanMessage

query = "What is 3 * 12? Also, what is 11 + 49?"

messages = [HumanMessage(query)] 

ai_msg = llm_with_tools.invoke(messages)

print(ai_msg.tool_calls)

messages.append(ai_msg) #append the ai message to the messages list 

[{'name': 'multiply', 'args': {'a': 3, 'b': 12}, 'id': 'call_0_894da76a-ab59-4f56-b484-f896c5ae9695', 'type': 'tool_call'}, {'name': 'add', 'args': {'a': 11, 'b': 49}, 'id': 'call_1_7f30fecb-2fa1-4dbe-b5f0-10d9304817ea', 'type': 'tool_call'}]


if we invoke a LangChain Tool with a ToolCall, we'll automatically get back a ToolMessage that can be fed back to the model:

In [62]:
for tool_call in ai_msg.tool_calls:
    selected_tool = {"add": add, "multiply": multiply}[tool_call["name"].lower()]
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
input_data = [
    tool_call,
    tool_msg
]
messages

[HumanMessage(content='What is 3 * 12? Also, what is 11 + 49?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_0_894da76a-ab59-4f56-b484-f896c5ae9695', 'function': {'arguments': '{"a": 3, "b": 12}', 'name': 'multiply'}, 'type': 'function', 'index': 0}, {'id': 'call_1_7f30fecb-2fa1-4dbe-b5f0-10d9304817ea', 'function': {'arguments': '{"a": 11, "b": 49}', 'name': 'add'}, 'type': 'function', 'index': 1}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 228, 'total_tokens': 276, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'prompt_cache_hit_tokens': 192, 'prompt_cache_miss_tokens': 36}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_3a5770e1b4', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-6733461b-794c-43d3-8c59-5522d4844d90-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 12}, 'id': 'call_0_894da76a-ab59-4f56-b484-f896c5ae

llm_with_tools.invoke 测试不成功

In [56]:
llm_with_tools.invoke(messages) #invoke the llm with the messages list

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 0, 'prompt_tokens': 286, 'total_tokens': 286, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'prompt_cache_hit_tokens': 192, 'prompt_cache_miss_tokens': 94}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_3a5770e1b4', 'finish_reason': 'stop', 'logprobs': None}, id='run-97d18242-2b3c-4087-81ee-12243b386339-0', usage_metadata={'input_tokens': 286, 'output_tokens': 0, 'total_tokens': 286, 'input_token_details': {}, 'output_token_details': {}})

In [66]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个智能助手，负责将工具调用结果转换为自然语言。"),
    ("human", "请将以下工具调用结果转换为自然语言：{input}")
])

# 创建转换链
chain = prompt | llm | StrOutputParser()
    
response = chain.invoke({"input": messages})

print(response)

3 乘以 12 的结果是 36，11 加上 49 的结果是 60。


Chain & .Pipe

In [87]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("tell me a joke about {topic}")
chain = prompt | llm | StrOutputParser()

analysis_prompt = ChatPromptTemplate.from_template("is this a funny joke? {joke}")
composed_chain_with_lambda = (
    chain
    | (lambda input: {"joke": input})
    | analysis_prompt
    | llm
    | StrOutputParser()
)

composed_chain_with_lambda.invoke({"topic": "beets"})

"Haha, that's a classic! It's definitely a lighthearted and punny joke, perfect for a quick chuckle. The beet turning red because it saw the salad dressing is a clever play on words, and it’s the kind of joke that works well in casual settings or with people who appreciate a good vegetable pun. 😄🥗"

In [88]:
chain.invoke({"topic": "beets"})

"Sure! Here's a beet-related joke for you:\n\nWhy did the beet turn red?\n\nBecause it saw the salad dressing! 🥗😄"

In [89]:
from langchain_core.runnables import RunnableParallel

composed_chain_with_pipe = (
    RunnableParallel({"joke": chain})
    .pipe(analysis_prompt)
    .pipe(llm)
    .pipe(StrOutputParser())
)

composed_chain_with_pipe.invoke({"topic": "battlestar galactica"})

'Yes, that\'s a funny joke! It works on multiple levels, which is what makes it clever. The pun on "issues" ties into both the Cylons\' struggle with identity (a major theme in *Battlestar Galactica*) and the literal idea of different Cylon models or "issues." Plus, the idea of a Cylon going to therapy is inherently amusing. It’s a great mix of fandom humor and wordplay! 😄'